# 📈 Stock Price Prediction – CSE247 Applied Machine Learning

**Lovely Professional University | School of Computer Science and Engineering**

This notebook demonstrates the complete Machine Learning pipeline for stock price prediction:
1. Problem Statement
2. Data Loading & Preprocessing
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Model Implementation (Linear Regression, Decision Tree, Random Forest)
6. Model Evaluation & Comparison
7. Hyperparameter Tuning (GridSearchCV)
8. Final Prediction

## 1. Problem Statement

**Objective:** Predict the **next day's closing price** of a given stock using historical OHLCV data and derived technical indicators.

**Type:** Supervised Regression

**Target Variable:** Next day's closing price (`Target = Close.shift(-1)`)

**Features Used:**
- Open, High, Low, Close prices
- 10-day Simple Moving Average (SMA_10)
- 50-day Simple Moving Average (SMA_50)
- Daily Return (%)
- 10-day Volatility (rolling std of daily return)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('All libraries imported successfully ✅')

## 2. Data Loading & Preprocessing

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
TICKER = 'AAPL'
START  = '2020-01-01'
END    = '2026-05-01'

stock = yf.Ticker(TICKER)
data  = stock.history(start=START, end=END)

print(f'Loaded {len(data)} trading days for {TICKER}')
print(f'Date range: {data.index[0].date()} → {data.index[-1].date()}')
data.head()

In [ ]:
# ── Data Info & Null Check ────────────────────────────────────────────────────
print('Shape:', data.shape)
print('\nNull values:')
print(data.isnull().sum())
print('\nBasic Statistics:')
data[['Open','High','Low','Close','Volume']].describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Historical Close Price ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(data.index, data['Close'], color='royalblue', linewidth=1.2, label='Close Price')
ax.set_title(f'{TICKER} – Historical Closing Price', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Moving Averages ───────────────────────────────────────────────────────────
eda = data.copy()
eda['SMA_10'] = eda['Close'].rolling(10).mean()
eda['SMA_50'] = eda['Close'].rolling(50).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(eda.index, eda['Close'],  color='royalblue', linewidth=1,   label='Close')
ax.plot(eda.index, eda['SMA_10'], color='orange',    linewidth=1.5, linestyle='--', label='SMA 10')
ax.plot(eda.index, eda['SMA_50'], color='green',     linewidth=1.5, linestyle='--', label='SMA 50')
ax.set_title('Close Price with Simple Moving Averages', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Volume & Daily Returns ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Volume
axes[0].bar(data.index, data['Volume'], color='steelblue', alpha=0.6, width=2)
axes[0].set_title('Daily Trading Volume'); axes[0].set_xlabel('Date')
axes[0].set_ylabel('Volume')

# Returns distribution
daily_ret = data['Close'].pct_change().dropna() * 100
axes[1].hist(daily_ret, bins=80, color='royalblue', alpha=0.7, edgecolor='white')
axes[1].axvline(daily_ret.mean(), color='red', linestyle='--', label=f'Mean: {daily_ret.mean():.2f}%')
axes[1].set_title('Daily Return Distribution')
axes[1].set_xlabel('Daily Return (%)'); axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout(); plt.show()

print(f'Mean daily return: {daily_ret.mean():.4f}%')
print(f'Std daily return:  {daily_ret.std():.4f}%')
print(f'Max daily return:  {daily_ret.max():.4f}%')
print(f'Min daily return:  {daily_ret.min():.4f}%')

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
corr_df = data[['Open','High','Low','Close','Volume']].copy()
corr_df['SMA_10'] = corr_df['Close'].rolling(10).mean()
corr_df['SMA_50'] = corr_df['Close'].rolling(50).mean()
corr_df = corr_df.dropna()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Feature Engineering

In [ ]:
# ── Build Feature Matrix ──────────────────────────────────────────────────────
df = data.copy()

df['SMA_10']       = df['Close'].rolling(10).mean()
df['SMA_50']       = df['Close'].rolling(50).mean()
df['Daily_Return'] = df['Close'].pct_change()
df['Volatility']   = df['Daily_Return'].rolling(10).std()
df['Target']       = df['Close'].shift(-1)   # Next day's close (label)

df_clean = df.dropna()

FEATURES = ['Close', 'Open', 'High', 'Low', 'SMA_10', 'SMA_50', 'Daily_Return', 'Volatility']
X = df_clean[FEATURES]
y = df_clean['Target']

print('Feature matrix shape:', X.shape)
print('Target shape:         ', y.shape)
X.head()

In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False   # Time series – no shuffling!
)
print(f'Training samples: {len(X_train)}')
print(f'Testing  samples: {len(X_test)}')

# Feature scaling (for Linear Regression)
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

## 5. Model Implementation

We train **three** models:
1. **Linear Regression** – baseline linear model
2. **Decision Tree Regressor** – non-linear, interpretable tree
3. **Random Forest Regressor** – ensemble of decision trees

In [ ]:
# ── Model 1: Linear Regression ────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
lr_preds = lr.predict(X_test_sc)

print('Linear Regression Coefficients:')
for feat, coef in zip(FEATURES, lr.coef_):
    print(f'  {feat:20s}: {coef:.4f}')

In [ ]:
# ── Model 2: Decision Tree Regressor ──────────────────────────────────────────
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)
print('Decision Tree trained ✅  (max_depth=5)')

In [ ]:
# ── Model 3: Random Forest Regressor ─────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
print('Random Forest trained ✅  (n_estimators=100, max_depth=8)')

## 6. Model Evaluation & Comparison

In [ ]:
# ── Metrics Helper ────────────────────────────────────────────────────────────
def evaluate(name, y_true, y_pred):
    return {
        'Model': name,
        'MAE':   round(mean_absolute_error(y_true, y_pred), 4),
        'RMSE':  round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'R²':    round(r2_score(y_true, y_pred), 4)
    }

results = pd.DataFrame([
    evaluate('Linear Regression', y_test, lr_preds),
    evaluate('Decision Tree',     y_test, dt_preds),
    evaluate('Random Forest',     y_test, rf_preds),
])

print('Model Comparison (Test Set):')
display(results.set_index('Model'))

In [ ]:
# ── Predicted vs Actual Plots ─────────────────────────────────────────────────
model_results = [
    ('Linear Regression', lr_preds, 'orange'),
    ('Decision Tree',     dt_preds, 'green'),
    ('Random Forest',     rf_preds, 'red'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, (name, preds, color) in zip(axes, model_results):
    ax.plot(y_test.index, y_test.values, label='Actual',    color='royalblue', linewidth=1.2)
    ax.plot(y_test.index, preds,         label='Predicted', color=color,       linewidth=1.2, linestyle='--')
    r2  = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    ax.set_title(f'{name}\nR²={r2:.4f}  MAE=${mae:.2f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Date'); ax.set_ylabel('Price (USD)')
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Predicted vs Actual Closing Price – All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Cross-Validation Scores ───────────────────────────────────────────────────
cv_models = [
    ('Linear Regression', LinearRegression(), X_train_sc),
    ('Decision Tree',     DecisionTreeRegressor(max_depth=5, random_state=42), X_train),
    ('Random Forest',     RandomForestRegressor(n_estimators=50, random_state=42), X_train),
]

print('5-Fold Cross-Validation R² Scores:')
for name, model, Xtr in cv_models:
    scores = cross_val_score(model, Xtr, y_train, cv=5, scoring='r2')
    print(f'  {name:22s}:  Mean = {scores.mean():.4f}  Std = {scores.std():.4f}')

## 7. Hyperparameter Tuning (GridSearchCV)

In [ ]:
# ── GridSearchCV on Random Forest ─────────────────────────────────────────────
param_grid = {
    'n_estimators':      [50, 100, 200],
    'max_depth':         [5, 8, 12],
    'min_samples_split': [2, 5],
}

rf_base    = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(rf_base, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print('\nBest Parameters:', grid_search.best_params_)
print('Best CV R²:     ', round(grid_search.best_score_, 4))

In [ ]:
# ── Evaluate Tuned Model ──────────────────────────────────────────────────────
best_rf    = grid_search.best_estimator_
tuned_preds = best_rf.predict(X_test)

tuned_eval = evaluate('Tuned Random Forest', y_test, tuned_preds)
all_results = pd.DataFrame([
    evaluate('Linear Regression', y_test, lr_preds),
    evaluate('Decision Tree',     y_test, dt_preds),
    evaluate('Random Forest',     y_test, rf_preds),
    tuned_eval
])

print('\nFinal Model Comparison (including tuned model):')
display(all_results.set_index('Model'))

In [ ]:
# ── Tuned RF Prediction Plot ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test.index, y_test.values,  label='Actual',            color='royalblue', linewidth=1.5)
ax.plot(y_test.index, tuned_preds, label='Tuned RF Predicted', color='purple',    linewidth=1.5, linestyle='--')
ax.set_title(f'Tuned Random Forest – Predicted vs Actual (R²={tuned_eval["R²"]:.4f})', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price (USD)')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────────
importances = pd.Series(best_rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importance – Tuned Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
ax.grid(True, linestyle='--', alpha=0.5, axis='x')
plt.tight_layout(); plt.show()

## 8. Final Prediction – Tomorrow's Closing Price

In [ ]:
# ── Tomorrow's Prediction ─────────────────────────────────────────────────────
latest = data.copy()
latest['SMA_10']       = latest['Close'].rolling(10).mean()
latest['SMA_50']       = latest['Close'].rolling(50).mean()
latest['Daily_Return'] = latest['Close'].pct_change()
latest['Volatility']   = latest['Daily_Return'].rolling(10).std()

current = latest[FEATURES].iloc[-1:]
next_pred  = best_rf.predict(current)[0]
last_close = latest['Close'].iloc[-1]
diff       = next_pred - last_close
diff_pct   = (diff / last_close) * 100

print(f'Ticker:                    {TICKER}')
print(f'Last Close Price:          ${last_close:.2f}')
print(f'Predicted Next Close:      ${next_pred:.2f}')
print(f'Expected Change:           ${diff:+.2f}  ({diff_pct:+.2f}%)')
print(f'Model Used:                Tuned Random Forest')
print()
print('⚠️  Disclaimer: Educational project only. Do NOT use for real trading.')